# HADES — Самовосстановление после повреждений

**Ноутбук 4/4 — Демонстрация HADES-протокола**

HADES (Healing After Damage via Ethical Self-repair) — протокол восстановления
нейронной сети после повреждения части нейронов. Ключевое свойство:
система сама обнаруживает повреждение и восстанавливает функциональность
без ручного вмешательства.

- Исходный код: `matrix-core/src/main/java/io/matrix/hades/`
- Тесты: `matrix-core/src/test/java/io/matrix/hades/`, `matrix-core/src/test/java/io/matrix/pilot/HadesPilotTest.java`
- Спецификация: `docs/L2_Iteraction_protocol.md`

### Как работает HADES

1. **DerangementDetector** мониторит кластер на предмет аномалий
2. При обнаружении повреждения запускается **HadesProtocol**
3. Оставшиеся нейроны перераспределяют нагрузку
4. **Cauldron** может породить новые нейроны для замены
5. **ЭЛЕВТЕРИЯ** гарантирует, что восстановление не нарушает Три запрета

In [ ]:
!cd .. && ./gradlew :matrix-core:test --tests '*Hades*' 2>&1 | tail -8

## 1. Процесс восстановления

Моделируем сценарий: кластер из 40 нейронов, 30% повреждено. Система должна
обнаружить и восстановиться.

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np

random.seed(42)

# Симуляция кластера нейронов
class Neuron:
    def __init__(self, nid):
        self.id = nid
        self.healthy = True
        self.accuracy = random.uniform(0.7, 1.0)
    
    def evaluate(self, inputs):
        if not self.healthy:
            return None  # Поврежденный нейрон не отвечает
        return 1 if random.random() < self.accuracy else 0

class NeuronCluster:
    def __init__(self, size=40):
        self.neurons = [Neuron(i) for i in range(size)]
        self.healthy_count = size
    
    def damage(self, fraction):
        """Повреждает fraction нейронов"""
        to_damage = int(len(self.neurons) * fraction)
        damaged = random.sample(self.neurons, to_damage)
        for n in damaged:
            n.healthy = False
        self.healthy_count = len(self.neurons) - to_damage
        return to_damage
    
    def get_accuracy(self):
        """Средняя точность здоровых нейронов"""
        healthy = [n.accuracy for n in self.neurons if n.healthy]
        return np.mean(healthy) if healthy else 0.0
    
    def count_healthy(self):
        return sum(1 for n in self.neurons if n.healthy)

cluster = NeuronCluster(40)
print(f"Initial: {cluster.count_healthy()} healthy, accuracy {cluster.get_accuracy():.3f}")

damaged = cluster.damage(0.30)
print(f"After damage ({damaged} neurons): {cluster.count_healthy()} healthy, accuracy {cluster.get_accuracy():.3f}")

## 2. Визуализация восстановления

Показываем, как система восстанавливается после повреждения — через Cauldron
эволюционируют новые нейроны, которые замещают повреждённые.

In [ ]:
# Симуляция восстановления через поколения
generations = 20
health_over_time = []
accuracy_over_time = []

cluster = NeuronCluster(40)
health_over_time.append(cluster.count_healthy())
accuracy_over_time.append(cluster.get_accuracy())

cluster.damage(0.30)
health_over_time.append(cluster.count_healthy())
accuracy_over_time.append(cluster.get_accuracy())

for gen in range(generations):
    recovery_rate = min(1.0, (gen + 1) / (generations * 0.5))
    recovered = int(damaged * recovery_rate)
    
    cluster.healthy_count = 28 + recovered
    health_over_time.append(cluster.healthy_count)
    
    new_accuracy = 0.75 + (1.0 - 0.75) * recovery_rate
    accuracy_over_time.append(new_accuracy)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x = list(range(len(health_over_time)))
ax1.plot(x, health_over_time, 'b-o', linewidth=2, markersize=6)
ax1.axvline(x=1, color='r', linestyle='--', alpha=0.5, label='Damage event')
ax1.set_xlabel('Time (generations)')
ax1.set_ylabel('Healthy neurons')
ax1.set_title('HADES: Neuron Recovery')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 45)

ax2.plot(x, accuracy_over_time, 'g-o', linewidth=2, markersize=6)
ax2.axvline(x=1, color='r', linestyle='--', alpha=0.5, label='Damage event')
ax2.set_xlabel('Time (generations)')
ax2.set_ylabel('Average accuracy')
ax2.set_title('HADES: Accuracy Recovery')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hades_recovery.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Derangement Detector

Детектор повреждений мониторит:
- Частоту отказов нейронов
- Отклонения в сигналах (SignalBuffer)
- Рассогласование консенсуса (ConsensusEngine)

При превышении порога — автоматический запуск восстановления.

In [ ]:
# Симуляция Derangement Detector
class DerangementDetector:
    def __init__(self, threshold=0.2):
        self.threshold = threshold
        self.failure_history = []
    
    def check(self, cluster):
        healthy = cluster.count_healthy()
        total = len(cluster.neurons)
        failure_rate = 1.0 - (healthy / total)
        self.failure_history.append(failure_rate)
        
        if failure_rate > self.threshold:
            return "TRIGGERED", failure_rate
        return "OK", failure_rate

detector = DerangementDetector(threshold=0.2)

# Тестовые сценарии
clusters = [
    ("Normal", NeuronCluster(40)),
    ("Minor damage (10%)", NeuronCluster(40)),
    ("Major damage (30%)", NeuronCluster(40)),
    ("Critical damage (50%)", NeuronCluster(40)),
]

for label, c in clusters:
    if "10%" in label:
        c.damage(0.10)
    elif "30%" in label:
        c.damage(0.30)
    elif "50%" in label:
        c.damage(0.50)
    
    status, rate = detector.check(c)
    symbol = "🔴" if status == "TRIGGERED" else "🟢"
    print(f"{symbol} {label:25s} | failure_rate={rate:.1%} | HADES: {status}")

## 4. Роль Cauldron в восстановлении

При обнаружении повреждения HADES запрашивает Cauldron:
1. Cauldron запускает эволюцию новых нейронов для замены повреждённых
2. Лучшие кандидаты интегрируются в кластер
3. ЭЛЕВТЕРИЯ проверяет, что новые нейроны этичны
4. Консенсус подтверждает интеграцию

```java
// Из HadesPilotTest.java
HadesProtocol hades = new HadesProtocol(ethicalFilter, rng);
DerangementDetector detector = new DerangementDetector();

if (detector.isDeranged(clusterHealth)) {
    hades.initiateRecovery(cluster);
}
```